# Interval Propagation version of 3d sim

In [ ]:
# main_3d_interval.ipynb - 3D interval reentry driver (with heat shield plots)

import os, math, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import constants
import AtmosphereModel
import interval_math
import math_3d

importlib.reload(constants)
importlib.reload(AtmosphereModel)
importlib.reload(interval_math)
importlib.reload(math_3d)

from interval_math import Interval, promote, scalar_times_interval

## small plotting plus interval helpers

In [ ]:
def iv_lo(iv): return float(iv.lo)
def iv_hi(iv): return float(iv.hi)
def iv_mid(iv): return float(iv.mid())
def iv_w(iv): return float(iv.width())

def arr_mid(iv_list): return np.array([iv_mid(v) for v in iv_list], dtype=float)
def arr_lo(iv_list):  return np.array([iv_lo(v)  for v in iv_list], dtype=float)
def arr_hi(iv_list):  return np.array([iv_hi(v)  for v in iv_list], dtype=float)
def arr_w(iv_list):   return np.array([iv_w(v)   for v in iv_list], dtype=float)

def plot_band(t, y_iv, ylabel, title):
    y_mid = arr_mid(y_iv)
    y_lo  = arr_lo(y_iv)
    y_hi  = arr_hi(y_iv)

    plt.figure(figsize=(10,4))
    plt.plot(t, y_mid)
    plt.fill_between(t, y_lo, y_hi, alpha=0.25)
    plt.xlabel("t [s]")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True)

def hull_many(intervals):
    out = None
    for iv in intervals:
        out = iv if out is None else out.hull(iv)
    return out if out is not None else Interval(0.0, 0.0)

def rad2deg(x): return x * 180.0 / math.pi


## fix the layer hull version of the 3D interval EOM

In [ ]:
def intv_eom_3d_hull(t, X, params, sigma_iv):
    """
    Like math_3d.intv_eom_3d, but correctly hulls aero across multiple layers
    and also returns rho,q hulls for diagnostics/heating.
    """
    r, phi, lam, V, gamma, chi = X
    m = float(params["mass_kg"])

    g = constants.intv_gravity(r)

    aero = math_3d.intv_aero_forces(
        r=r, V=V, gamma=gamma, chi=chi, sigma=sigma_iv, params=params
    )

    Dr     = hull_many([aero[k]["Dr"] for k in aero.keys()])
    Dtheta = hull_many([aero[k]["Dtheta"] for k in aero.keys()])
    Dphi   = hull_many([aero[k]["Dphi"] for k in aero.keys()])
    Lr     = hull_many([aero[k]["Lr"] for k in aero.keys()])
    Ltheta = hull_many([aero[k]["Ltheta"] for k in aero.keys()])
    Lphi   = hull_many([aero[k]["Lphi"] for k in aero.keys()])
    rho    = hull_many([aero[k]["rho"] for k in aero.keys()])
    q      = hull_many([aero[k]["q"]   for k in aero.keys()]) if "q" in next(iter(aero.values())) else None

    # EOM (interval version)
    r_dot   = V * Interval.sin(gamma)
    phi_dot = (V * Interval.cos(gamma) * Interval.sin(chi)) / r

    # guard cos(phi) near 0: you can clamp later if you want
    lam_dot = (V * Interval.cos(gamma) * Interval.cos(chi)) / (r * Interval.cos(phi))

    V_dot = (Dr + Lr) / m - g * Interval.sin(gamma)

    gamma_dot = (Ltheta / (m * V)) + (V / r - g / V) * Interval.cos(gamma)

    phi_tan = (Interval.sin(phi)) / (Interval.cos(phi))
    chi_dot = (Lphi / (m * V * Interval.cos(gamma))) + (V / r) * Interval.sin(chi) * phi_tan

    return {
        "r_dot": r_dot,
        "phi_dot": phi_dot,
        "lam_dot": lam_dot,
        "V_dot": V_dot,
        "gamma_dot": gamma_dot,
        "chi_dot": chi_dot,
        "rho": rho,
        "q": q,
    }


## Scenario config + initial interval state

### All measurments are taken reference to the NASA Apollo Capsule

In [ ]:
# Vehicle params expected by your interval aero code (mass_kg/ref_area_m2/CL_const/CD_const/nose_radius_m)
vehicle = {
    "mass_kg": 5560.0,
    "ref_area_m2": 12.0,
    "CL_const": 0.35,
    "CD_const": 1.15,
    "nose_radius_m": 0.232,
}

# Heat shield - as per an appolo mission dimension capsule
shield = constants.HeatShield(
    radius_m=1.9558,        # 3.9 meters radius for appolo capsule 
    nose_radius_m=1.0,   # must match nose radius used in Sutton–Graves
    num_rings=6,
    radial_exp=1.0
)

# Time setup
dt = 0.25
t0 = 0.0
t_final = 2000.0

# Initial conditions (add small uncertainty so intervals actually show bands)
h0 = Interval(80000.0, 80500.0)                    # altitude [m]
r0 = h0 + constants.RADIUS_EARTH                   # radius from Earth center [m]

phi0   = Interval(math.radians(20.0), math.radians(20.2))
lam0   = Interval(math.radians(30.0), math.radians(30.2))
V0     = Interval(7600.0, 7700.0)                  # [m/s]
gamma0 = Interval(math.radians(-6.0), math.radians(-5.5))
chi0   = Interval(math.radians(85.0), math.radians(87.0))

# Bank angle as an interval (small uncertainty)
sigma_iv = Interval(math.radians(-5.0), math.radians(5.0))

X0 = [r0, phi0, lam0, V0, gamma0, chi0]


# main sim loop

In [ ]:
def stop_condition(X):
    r, phi, lam, V, gamma, chi = X

    # Basic physical termination
    h = constants.intv_geometric_altitude(r)
    if h.hi <= 0.0:
        return True
    if V.hi <= 10.0:
        return True

    # Interval math domain guards for divisions used inside intv_eom_3d_hull
    # 1) V must stay strictly positive for terms like g/V and 1/(m*V)
    if V.lo <= 0.0:
        return True

    # 2) cos(phi) must not contain 0 for lambda_dot denominator r*cos(phi)
    #    and for tan(phi) via sin(phi)/cos(phi)
    cos_phi = Interval.cos(phi)
    if cos_phi.lo <= 0.0 <= cos_phi.hi:
        return True

    # 3) r must stay strictly positive for V/r
    if r.lo <= 0.0:
        return True

    return False


# Canonical Interval class used by constants.py for heat shield and constants utilities
IV = constants.Interval


def as_constants_interval(iv):
    # Convert interval_math.Interval into constants.Interval (punctual conversion of bounds)
    return IV(float(iv.lo), float(iv.hi))


# Patch HeatShield.update to enforce interval type consistency inside HeatShield arrays
def _safe_update(self, rho, V, dt):
    qdot_stag = self.stagnation_qdot(rho, V)

    for i, r in enumerate(self.r_centers):
        shape = self.radial_shape_factor(r)

        # Construct using the same Interval class as self.qdot[i]
        shape_iv = type(self.qdot[i])(shape, shape)

        self.qdot[i] = qdot_stag * shape_iv
        self.Q[i] = self.Q[i] + self.qdot[i] * dt


# Monkey patch once
constants.HeatShield.update = _safe_update


### actually run the code and make the trajectories

In [ ]:


# histories
t_hist = []
r_hist, h_hist = [], []
phi_hist, lam_hist = [], []
V_hist, gamma_hist, chi_hist = [], [], []
rho_hist, q_hist = [], []

qdot_rings_hist = []
Q_rings_hist = []
qdot_max_hist, Q_max_hist, qdot_mean_hist = [], [], []

X = X0[:]
t = t0

while t <= t_final and (not stop_condition(X)):
    r, phi, lam, V, gamma, chi = X
    h = constants.intv_geometric_altitude(r)

    # Extra safety: if any division domain violation would occur, stop before calling dynamics
    cos_phi = Interval.cos(phi)
    if V.lo <= 0.0 or (cos_phi.lo <= 0.0 <= cos_phi.hi) or (r.lo <= 0.0):
        break

    try:
        out = intv_eom_3d_hull(t, X, vehicle, sigma_iv)
    except ValueError as e:
        # Interval division failures and other domain errors should terminate cleanly
        print("Terminated due to interval domain violation:", e)
        break

    rho_for_shield = as_constants_interval(out["rho"])
    V_for_shield = as_constants_interval(V)

    # log state
    t_hist.append(t)
    r_hist.append(r)
    h_hist.append(h)
    phi_hist.append(phi)
    lam_hist.append(lam)
    V_hist.append(V)
    gamma_hist.append(gamma)
    chi_hist.append(chi)

    rho_hist.append(out["rho"])
    q_hist.append(out["q"] if out.get("q", None) is not None else IV(0.0, 0.0))

    # update heat shield
    shield.update(rho_for_shield, V_for_shield, dt)

    qdot_rings_hist.append(shield.qdot[:])
    Q_rings_hist.append(shield.Q[:])

    qdot_max_hist.append(hull_many(shield.qdot))
    Q_max_hist.append(hull_many(shield.Q))

    n = len(shield.qdot)

    # Mean scaling must use the same Interval class as hull_many output (interval_math.Interval)
    mean_scale_iv = type(qdot_max_hist[-1])(1.0 / n, 1.0 / n)
    qdot_mean_hist.append(qdot_max_hist[-1] * mean_scale_iv)

    # Euler step
    X = [
        r     + scalar_times_interval(dt, out["r_dot"]),
        phi   + scalar_times_interval(dt, out["phi_dot"]),
        lam   + scalar_times_interval(dt, out["lam_dot"]),
        V     + scalar_times_interval(dt, out["V_dot"]),
        gamma + scalar_times_interval(dt, out["gamma_dot"]),
        chi   + scalar_times_interval(dt, out["chi_dot"]),
    ]

    t += dt


t_arr = np.array(t_hist, dtype=float)
print("steps:", len(t_arr), "t_end:", t_arr[-1] if len(t_arr) else None)


## state plots with interval bands

In [ ]:
plot_band(t_arr, h_hist, "h [m]", "Altitude vs time (interval band)")
plot_band(t_arr, V_hist, "V [m/s]", "Speed vs time (interval band)")

# angles in deg
gamma_deg = [Interval(rad2deg(iv.lo), rad2deg(iv.hi)) for iv in gamma_hist]
chi_deg   = [Interval(rad2deg(iv.lo), rad2deg(iv.hi)) for iv in chi_hist]

plot_band(t_arr, gamma_deg, "gamma [deg]", "Flight path angle vs time (interval band)")
plot_band(t_arr, chi_deg, "chi [deg]", "Heading angle vs time (interval band)")
plt.show()


## uncertainty growth

In [ ]:
def plot_width(t, y_iv, ylabel, title):
    plt.figure(figsize=(10,4))
    plt.plot(t, arr_w(y_iv))
    plt.xlabel("t [s]")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True)

plot_width(t_arr, h_hist, "width(h) [m]", "Altitude interval width vs time")
plot_width(t_arr, V_hist, "width(V) [m/s]", "Speed interval width vs time")
plot_width(t_arr, gamma_hist, "width(gamma) [rad]", "Gamma interval width vs time")
plot_width(t_arr, chi_hist, "width(chi) [rad]", "Chi interval width vs time")
plt.show()

### atmosphere + dynamic pressure bands

In [ ]:
plot_band(t_arr, rho_hist, "rho [kg/m^3]", "Density vs time (interval band)")
plot_band(t_arr, q_hist, "q [Pa]", "Dynamic pressure vs time (interval band)")
plt.show()

plot_width(t_arr, rho_hist, "width(rho)", "Density interval width vs time")
plot_width(t_arr, q_hist, "width(q)", "Dynamic pressure interval width vs time")
plt.show()


### Heating: qdot(t) and Q(t) bands + widths

In [ ]:
plot_band(t_arr, qdot_max_hist, "qdot_max [W/m^2]", "Peak heat flux vs time (interval band)")
plot_band(t_arr, Q_max_hist, "Q_max [J/m^2]", "Peak integrated heat load vs time (interval band)")
plt.show()

plot_width(t_arr, qdot_max_hist, "width(qdot_max)", "Peak heat flux interval width vs time")
plot_width(t_arr, Q_max_hist, "width(Q_max)", "Peak heat load interval width vs time")
plt.show()

# Plot per-ring heat flux (mid + band) for first few rings
num_rings = shield.num_rings
for i in range(num_rings):
    ring_i = [qdot_rings_hist[k][i] for k in range(len(t_arr))]
    plot_band(t_arr, ring_i, f"qdot_ring[{i}] [W/m^2]", f"Ring {i} heat flux vs time (interval band)")
plt.show()


### ground track (phi, lambda) midpoints

In [ ]:
phi_mid_deg = np.array([rad2deg(iv_mid(v)) for v in phi_hist], dtype=float)
lam_mid_deg = np.array([rad2deg(iv_mid(v)) for v in lam_hist], dtype=float)

plt.figure(figsize=(7,6))
plt.plot(lam_mid_deg, phi_mid_deg)
plt.xlabel("longitude [deg]")
plt.ylabel("latitude [deg]")
plt.title("Ground track (midpoint trajectory)")
plt.grid(True)
plt.show()


### 3D trajectory (ECEF-ish) using midpoints

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

r_mid = arr_mid(r_hist)
phi_mid = arr_mid(phi_hist)
lam_mid = arr_mid(lam_hist)

x = r_mid * np.cos(phi_mid) * np.cos(lam_mid)
y = r_mid * np.cos(phi_mid) * np.sin(lam_mid)
z = r_mid * np.sin(phi_mid)

fig = plt.figure(figsize=(7,7))
ax = fig.add_subplot(111, projection="3d")
ax.plot(x, y, z)
ax.set_title("3D trajectory (midpoint in Earth-centered coordinates)")
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_zlabel("z [m]")
plt.show()


## dump a trajectiry df for RL features 

In [ ]:
df = pd.DataFrame({
    "t_s": t_arr,

    "h_lo": arr_lo(h_hist),
    "h_mid": arr_mid(h_hist),
    "h_hi": arr_hi(h_hist),

    "V_lo": arr_lo(V_hist),
    "V_mid": arr_mid(V_hist),
    "V_hi": arr_hi(V_hist),

    "gamma_lo_deg": np.array([rad2deg(iv_lo(v)) for v in gamma_hist]),
    "gamma_mid_deg": np.array([rad2deg(iv_mid(v)) for v in gamma_hist]),
    "gamma_hi_deg": np.array([rad2deg(iv_hi(v)) for v in gamma_hist]),

    "rho_lo": arr_lo(rho_hist),
    "rho_mid": arr_mid(rho_hist),
    "rho_hi": arr_hi(rho_hist),

    "q_lo": arr_lo(q_hist),
    "q_mid": arr_mid(q_hist),
    "q_hi": arr_hi(q_hist),

    "qdot_max_lo": arr_lo(qdot_max_hist),
    "qdot_max_mid": arr_mid(qdot_max_hist),
    "qdot_max_hi": arr_hi(qdot_max_hist),

    "Q_max_lo": arr_lo(Q_max_hist),
    "Q_max_mid": arr_mid(Q_max_hist),
    "Q_max_hi": arr_hi(Q_max_hist),
})

df.head(), df.shape


## Save to csv

In [ ]:
import csv
from pathlib import Path

# -------------------------------
# Output directory (guaranteed)
# -------------------------------
data_folder = Path("data_intv") / "data_intv"
data_folder.mkdir(parents=True, exist_ok=True)

run_number = 1
traj_file = data_folder / f"interval_trajectory_{run_number}.csv"

# -------------------------------
# Write trajectory CSV
# -------------------------------
with open(traj_file, "w", newline="") as f:
    writer = csv.writer(f)

    writer.writerow([
        "step",
        "dt",
        "t",
        "r_lo", "r_hi",
        "h_lo", "h_hi",
        "V_lo", "V_hi",
        "gamma_lo", "gamma_hi",
        "chi_lo", "chi_hi",
        "rho_lo", "rho_hi",
    ])

    for k in range(len(t_hist)):
        writer.writerow([
            k,
            dt,
            t_hist[k],
            r_hist[k].lo, r_hist[k].hi,
            h_hist[k].lo, h_hist[k].hi,
            V_hist[k].lo, V_hist[k].hi,
            gamma_hist[k].lo, gamma_hist[k].hi,
            chi_hist[k].lo, chi_hist[k].hi,
            rho_hist[k].lo, rho_hist[k].hi,
        ])

print(f"Saved trajectory data to {traj_file.resolve()}")


### separate csv for the heat data

In [ ]:
import csv
from pathlib import Path

# -------------------------------
# Heat shield CSV path
# -------------------------------
heat_file = data_folder / f"interval_shield_{run_number}.csv"

with open(heat_file, "w", newline="") as f:
    writer = csv.writer(f)

    header = [
        "step",
        "dt",
        "qdot_max_lo", "qdot_max_hi",
        "Q_max_lo", "Q_max_hi",
        "qdot_mean_lo", "qdot_mean_hi",
    ]

    # Per-ring columns
    for i in range(shield.num_rings):
        header += [
            f"ring{i}_qdot_lo", f"ring{i}_qdot_hi",
            f"ring{i}_Q_lo", f"ring{i}_Q_hi",
        ]

    writer.writerow(header)

    for k in range(len(t_hist)):
        row = [
            k,
            dt,
            shield.qdot_max().lo, shield.qdot_max().hi,
            shield.Q_max().lo, shield.Q_max().hi,
            shield.qdot_mean().lo, shield.qdot_mean().hi,
        ]

        for i in range(shield.num_rings):
            row += [
                shield.qdot[i].lo, shield.qdot[i].hi,
                shield.Q[i].lo, shield.Q[i].hi,
            ]

        writer.writerow(row)

print(f"Saved heat shield data to {heat_file.resolve()}")
